In [6]:
from tqdm import tqdm
from typing import List, Tuple


import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch_geometric.datasets import MoleculeNet
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
matplotlib.use("Agg")  # backend non interactif : sauvegarde sans affichage


In [13]:


TASK_NAMES = ["NR-AR", "NR-AR-LBD", "NR-AhR", "NR-ER", "NR-ER-LBD",
              "NR-PPAR-gamma", "NR-Aromatase", "SR-ARE", "SR-ATAD5",
              "SR-HSE", "SR-MMP", "SR-p53"]

FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Generateur Morgan instancie UNE fois (nouvelle API RDKit, plus de deprecation)
MORGAN_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


# ------------------------------------------------------------- chargement dataset

def load_tox21(root: str = "data/") -> MoleculeNet:
    """Tox21 via PyG : telecharge + cache au premier appel (dossier data/, ignore par git).
    Chaque molecule : x (atomes), edge_index (liaisons), y (12 taches, NaN = non teste),
    smiles."""
    dataset = MoleculeNet(root=root, name="Tox21")
    print(f"Tox21 : {len(dataset)} molecules")
    return dataset


# ------------------------------------------------------------------ scaffold split

def murcko_scaffold_smiles(data) -> str:
    """SMILES canonique du scaffold Bemis-Murcko : la clef de regroupement."""
    mol = Chem.MolFromSmiles(data.smiles)
    if mol is None:
        return ""
    return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))


def scaffold_split(dataset, frac_train=0.8, frac_val=0.1, frac_test=0.1,
                   seed: int = 0):
    """Scaffold split : toutes les molecules d'un meme scaffold dans le MEME split
    -> pas de fuite structurelle (contrairement au split aleatoire).
    Groupes tries par taille decroissante, remplissage train -> val -> test,
    ordre melange par seed -> split deterministe."""
    assert abs(frac_train + frac_val + frac_test - 1.0) < 1e-6
    rng = torch.Generator().manual_seed(seed)

    scaffold_to_idx = {}
    for i in range(len(dataset)):
        scaffold_to_idx.setdefault(murcko_scaffold_smiles(dataset[i]), []).append(i)

    # Le seed doit mélanger l'ordre DES GROUPES (pas leur interieur) :
    # permuter l'interieur d'un groupe ne change pas le split de ses membres,
    # d'ou des results rigoureusement identiques pour tous les seeds (bug fixe).
    groups = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    order = torch.randperm(len(groups), generator=rng).tolist()
    groups = [groups[k] for k in order]

    n = len(dataset)
    train, val, test = [], [], []
    for group in groups:
        for idx in group:
            if len(train) < frac_train * n:
                train.append(idx)
            elif len(val) < frac_val * n:
                val.append(idx)
            else:
                test.append(idx)
    return train, val, test


# ---------------------------------------------------------------- featurisation

def smiles_to_fingerprint(smiles: str, radius: int = 2, n_bits: int = 2048) -> np.ndarray:
    """ECFP4 : SMILES -> vecteur binaire (n_bits,). Sparse, sans fit -> pas de fuite."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES invalide : {smiles!r}")
    fp = MORGAN_GEN.GetFingerprint(mol)
    return np.asarray(fp, dtype=np.float32)


def build_feature_matrix(dataset) -> np.ndarray:
    """Tout le dataset -> X (n_molecules, n_bits). Une seule fois, reutilise partout."""
    X = np.stack([smiles_to_fingerprint(d.smiles) for d in dataset])
    print(f"Features : X {X.shape}, densite {X.mean():.4f}")
    return X


def stack_labels(dataset) -> np.ndarray:
    """y de PyG (n, 1, 12) -> Y (n, 12) avec NaN conserves."""
    return torch.stack([d.y for d in dataset]).squeeze(1).numpy()


# --------------------------------------------------------------------- entrainement

def train_svm_per_task(X: np.ndarray,
                       Y: np.ndarray,
                       idx_train: list,
                       C: float = 1.0):
    """Un LinearSVC par tache, sur les lignes annotees uniquement.
    class_weight='balanced' compense le desequilibre mesure dans l'EDA.
    Decision_function servira pour l'AUC -> pas de probability=True (Platt couteux)."""
    models = []
    for t in range(Y.shape[1]):
        mask_tr = ~np.isnan(Y[idx_train, t])
        Xtr = X[idx_train][mask_tr]
        ytr = Y[idx_train, t][mask_tr].astype(int)
        clf = LinearSVC(C=C, class_weight="balanced", max_iter=10_000)
        clf.fit(Xtr, ytr)
        models.append(clf)
    return models


# ---------------------------------------------------------------------- evaluation

def evaluate_svm(models, X: np.ndarray, Y: np.ndarray, idx) -> dict:
    """AUC par tache sur les lignes annotees du split, + moyenne.
    AUC = metrique de ranking : les marges signees suffisent."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx, t])
        if mask.sum() < 2 or len(np.unique(Y[idx, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        scores = clf.decision_function(X[idx][mask])
        aucs.append(roc_auc_score(Y[idx, t][mask].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}




def auc_for_task(clf, X, Y, idx_val, t) -> float:
    mask = ~np.isnan(Y[idx_val, t])
    if mask.sum() < 2 or len(np.unique(Y[idx_val, t][mask])) < 2:
        return np.nan
    return roc_auc_score(Y[idx_val, t][mask].astype(int),
                         clf.decision_function(X[idx_val][mask]))

def tanimoto_kernel(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Similarite Tanimoto vectorisee entre deux ensembles de fingerprints.
    K(i,j) = |Ai & Bj| / |Ai | Bj| - |Ai & Bj|   (bits communs / union des bits)"""
    inter = A @ B.T
    norm_a = A.sum(axis=1, keepdims=True)
    norm_b = B.sum(axis=1, keepdims=True).T
    union = norm_a + norm_b - inter
    return inter / union



def train_svm_tanimoto(G_full, Y, idx_train, C: float = 1.0):
    """Le masque NaN s'applique aux lignes ET colonnes via np.ix_."""
    models = []
    for t in range(Y.shape[1]):
        rows = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        G_tt = G_full[np.ix_(rows, rows)]        # Gram train masquee, symetrique
        clf = SVC(kernel="precomputed", C=C, class_weight="balanced")
        clf.fit(G_tt, Y[rows, t].astype(int))
        models.append(clf)
    return models


def evaluate_svm_tanimoto(models, G_full, Y, idx_train, idx_test) -> dict:
    aucs = []
    for t, clf in enumerate(models):
        rows_tr = np.array([i for i in idx_train if not np.isnan(Y[i, t])])
        rows_te = np.array([i for i in idx_test  if not np.isnan(Y[i, t])])
        G_te = G_full[np.ix_(rows_te, rows_tr)]  # (n_test, n_train) — largeur du fit
        scores = clf.decision_function(G_te)
        aucs.append(roc_auc_score(Y[rows_te, t].astype(int), scores))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}

# ------------------------------------------------------------------------- pipeline

def plot_auc_per_task(model_name : str,auc_matrix: np.ndarray, out: Path = FIG_DIR / "auc_per_task.png"):
    """Barres horizontales : AUC par tache (moyenne +/- std sur les seeds).
    Sauvegarde dans results/figures (versionne) + retourne la figure."""
    aucs = np.nanmean(auc_matrix, axis=0)
    stds = np.nanstd(auc_matrix, axis=0)
    order = np.argsort(aucs)  # triees de la pire a la meilleure

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(np.array(TASK_NAMES)[order], aucs[order],
            xerr=stds[order], color="#4C72B0", alpha=1)
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Hasard (0.5)")
    ax.axvline(np.mean(aucs),color="orange", linestyle="--", linewidth=1, label=" Mean of models")
    ax.set_xlim(0.4, 1)
    ax.set_xlabel("AUC ROC (test, scaffold split)")
    ax.set_title(f"Baseline {model_name} + ECFP4 - AUC par tache")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    print(f"Mean  : {np.mean(aucs)}, std : {np.std(aucs)}")
    print(f"Figure sauvegardee : {out}")
    return fig


# ------------------------------------------------------------------------- pipeline

In [14]:
import lightgbm as lgb


def train_lgbm_per_task(X, Y, idx_train) -> list:
    """12 LGBMClassifier, un par tache, sur les lignes annotees uniquement.
    Contrainte LGBM : pas de NaN dans y (on drope), mais X peut rester intact
    (ECFP est binaire, aucun NaN de toute facon).
    scale_pos_weight = n_negatifs / n_positifs : l'equivalent natif LGBM
    de class_weight='balanced', calcule PAR TACHE (le ratio varie)."""
    models = []
    for t in range(Y.shape[1]):
        mask = ~np.isnan(Y[idx_train, t])
        y_t = Y[idx_train, t][mask].astype(int)
        n_pos = y_t.sum()
        n_neg = len(y_t) - n_pos
        spw = n_neg / max(n_pos, 1)          # garde-fou si 0 positif

        clf = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            scale_pos_weight=spw,
            verbosity=-1,                    # silence les logs internes
            random_state=42,                 # LGBM est stochastique : fixe-le
        )
        clf.fit(X[idx_train][mask], y_t)
        models.append(clf)
    return models


def evaluate_lgbm_per_task(models, X, Y, idx_test) -> dict:
    """AUC par tache sur lignes annotees, via predict_proba (probabilites
    natives de LGBM — pas besoin du hack decision_function du SVM)."""
    aucs = []
    for t, clf in enumerate(models):
        mask = ~np.isnan(Y[idx_test, t])
        if mask.sum() < 2 or len(np.unique(Y[idx_test, t][mask])) < 2:
            aucs.append(np.nan)
            continue
        proba = clf.predict_proba(X[idx_test][mask])[:, 1]
        aucs.append(roc_auc_score(Y[idx_test, t][mask].astype(int), proba))
    aucs = np.array(aucs)
    return {"auc_per_task": aucs, "auc_mean": np.nanmean(aucs)}

In [15]:

def run(model_name : str ="SVM", seed_list=(0, 1, 2,3,4,5,6,7,8,9,10,11,20,45,67), C: float = 1.0):
    dataset = load_tox21()
    X = build_feature_matrix(dataset)
    G=tanimoto_kernel(X,X)
    Y = stack_labels(dataset)

    all_means, all_aucs = [], []
    for seed in tqdm(seed_list):
        train, val, test = scaffold_split(dataset, seed=seed)
        if model_name=="SVM":
            models = train_svm_tanimoto(G, Y, train, C=C)
            res = evaluate_svm_tanimoto(models, G, Y,train,test)
        elif model_name=="LGBM":
            m_lgbm = train_lgbm_per_task(X, Y, train)
            res= evaluate_lgbm_per_task(m_lgbm, X, Y, test)
        # print(f"seed={seed} | AUC moyenne test = {res['auc_mean']:.4f}")
        # print("  par tache :", np.round(res["auc_per_task"], 3))
        all_means.append(res["auc_mean"])
        all_aucs.append(res["auc_per_task"])

    m = np.array(all_means)
    # print(f"\nSVM lineaire : AUC moyenne = {m.mean():.4f} +/- {m.std():.4f}")

    # Tableau console lisible
    auc_matrix = np.stack(all_aucs)                      # (n_seeds, 12)
    # print("\n" + "-" * 50)
    # for name, a, s in zip(TASK_NAMES, np.nanmean(auc_matrix, 0), np.nanstd(auc_matrix, 0)):
    #     print(f"  {name:<14} {a:.3f} +/- {s:.3f}")
    # print("-" * 50)

    plot_auc_per_task(model_name,auc_matrix)
    # np.savetxt(FIG_DIR.parent / "svm_auc_per_task.csv",
    #            auc_matrix, delimiter=",", header=",".join(TASK_NAMES), comments="")
    print("Ecart-type",m.std())
    return m.mean()


#if __name__ == "__main__":
#   run()

In [16]:
seeds=[i for i in range(100)]
seeds_test=[1,2,3,4,5]
param_C=1
run(model_name="LGBM",seed_list=seeds_test, C=param_C)

Tox21 : 7823 molecules


[20:27:48] WARNING: not removing hydrogen atom without neighbors


Features : X (7823, 2048), densite 0.0146


100%|██████████| 5/5 [03:03<00:00, 36.71s/it]


Mean  : 0.7283076913752439, std : 0.05424231504603025
Figure sauvegardee : results\figures\auc_per_task.png
Ecart-type 0.024037195457687285


np.float64(0.7283076913752439)

In [ ]:

def op_score(c : float):
    seeds_params=[i for i in range(10)]
    auc_mean=run(seed_list=seeds_params, C=c)
    return auc_mean

def select_C()-> Tuple[list, list]:
    Val=[1,10]
    Res=[]
    for x in tqdm(Val) :
        Res.append(op_score(x))
    return Res,Val

In [ ]:
Res,Val=select_C()

In [34]:
print(Res)

[np.float64(0.7736428924107324), np.float64(0.7642006685387428)]
